In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import pandas as pd

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


In [2]:
# Load the .npz file
traj = np.load('/home/dataopske/Desktop/jav/data/raw/worldmove/worldmove_trajectories.npz', allow_pickle=True)
wards_full_gdf = pd.read_csv('/home/dataopske/Desktop/jav/data/processed/wards_full_gdf(with_spatial_temporal_gini).csv')
# See what's inside (keys like 'lon', 'lat', 'time', 'agent_id', etc.)
print("Keys in the file:", traj.files)

# Example: Print shapes of arrays (adjust keys based on output)
for key in traj.files:
    print(f"{key}: shape={traj[key].shape}, dtype={traj[key].dtype}")

Keys in the file: ['grid', 'poi', 'pop', 'traj']
grid: shape=(), dtype=object
poi: shape=(30, 48, 34), dtype=float32
pop: shape=(30, 48), dtype=float64
traj: shape=(104538, 48), dtype=int64


Create a Monday-Friday

In [3]:
import pandas as pd

# Create date range
calendar = pd.DataFrame({
    'date': pd.date_range(start='2025-10-01', end='2025-10-31', freq='D')
})

# Add day of week (0=Monday, 6=Sunday)
calendar['weekday'] = calendar['date'].dt.day_name()
calendar['is_weekend'] = calendar['weekday'].isin(['Saturday', 'Sunday'])
calendar.head()


,date,weekday,is_weekend
0,2025-10-01,Wednesday,False
1,2025-10-02,Thursday,False
2,2025-10-03,Friday,False
3,2025-10-04,Saturday,True
4,2025-10-05,Sunday,True


In [4]:
# Your original data
df = wards_full_gdf.copy()

# Add a constant key to both for merge
df['key'] = 1
calendar['key'] = 1

# Cross join
df_with_calendar = pd.merge(df, calendar, on='key').drop('key', axis=1)


In [5]:
df_with_calendar = df_with_calendar.sort_values(['ward', 'date', 'hour']).reset_index(drop=True)


Reasonable starting multipliers (apply to the weekday trips_per_hour values when you copy them to weekend dates):

Saturday: 0.6 — 0.8 (start with 0.7 as a central estimate)

Sunday: 0.4 — 0.6 (start with 0.5 as a central estimate)

Why these? Many cities show weekend service/ridership often falls in roughly the 40–80% band of weekday peaks depending on mode/area; Nairobi’s informal network is heterogeneous so these mid-range values are conservative but realistic given the literature and local notes. 
ITDP
+2

In [6]:
import numpy as np

# Assume df_with_calendar is your ward-hour-date table after the cross-join
# and has columns: 'date','weekday','trips_per_hour','ward','hour',...

df = df_with_calendar.copy()

# central estimate multipliers
sat_mult = 0.7
sun_mult = 0.5

# apply multipliers (create new column so original stays)
def apply_weekend_multiplier(row, sat_mult=0.7, sun_mult=0.5):
    wd = row['weekday']
    if wd == 'Saturday':
        return row['trips_per_hour'] * sat_mult
    elif wd == 'Sunday':
        return row['trips_per_hour'] * sun_mult
    else:
        return row['trips_per_hour']

df['trips_per_hour_adj'] = df.apply(apply_weekend_multiplier, axis=1, sat_mult=sat_mult, sun_mult=sun_mult)

# recompute normalized metrics
df['service_per_1k_pop_adj'] = df['trips_per_hour_adj'] / df['population_y'] * 1000
df['trips_per_person_per_hour_adj'] = df['trips_per_hour_adj'] / df['population_y']


Account for traffic during the weekends

In [7]:
candidates = [
    {'sat':0.6, 'sun':0.4},
    {'sat':0.7, 'sun':0.5},  # central
    {'sat':0.8, 'sun':0.6},
]

results = []
for c in candidates:
    tmp = df.copy()
    tmp['trips_per_hour_adj'] = tmp.apply(lambda r: r['trips_per_hour'] * (c['sat'] if r['weekday']=='Saturday' else (c['sun'] if r['weekday']=='Sunday' else 1.0)), axis=1)
    summary = tmp.groupby('weekday')['trips_per_hour_adj'].mean().reindex(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
    results.append({'mult':c, 'weekday_means': summary.to_dict()})

# inspect results list to see how weekend means change

In [8]:
results

[{'mult': {'sat': 0.6, 'sun': 0.4},
  'weekday_means': {'Monday': 756.1333333333333,
   'Tuesday': 756.1333333333333,
   'Wednesday': 756.1333333333333,
   'Thursday': 756.1333333333333,
   'Friday': 756.1333333333333,
   'Saturday': 453.68,
   'Sunday': 302.4533333333334}},
 {'mult': {'sat': 0.7, 'sun': 0.5},
  'weekday_means': {'Monday': 756.1333333333333,
   'Tuesday': 756.1333333333333,
   'Wednesday': 756.1333333333333,
   'Thursday': 756.1333333333333,
   'Friday': 756.1333333333333,
   'Saturday': 529.2933333333333,
   'Sunday': 378.06666666666666}},
 {'mult': {'sat': 0.8, 'sun': 0.6},
  'weekday_means': {'Monday': 756.1333333333333,
   'Tuesday': 756.1333333333333,
   'Wednesday': 756.1333333333333,
   'Thursday': 756.1333333333333,
   'Friday': 756.1333333333333,
   'Saturday': 604.9066666666668,
   'Sunday': 453.68}}]

In [9]:
print(df.head(5))

   Unnamed: 0   gid  pop2009   county              subcounty          ward  \
0         198  2078  43168.0  Nairobi  Kamukunji  Sub County  Airbase Ward   
1         199  2078  43168.0  Nairobi  Kamukunji  Sub County  Airbase Ward   
2         200  2078  43168.0  Nairobi  Kamukunji  Sub County  Airbase Ward   
3         198  2078  43168.0  Nairobi  Kamukunji  Sub County  Airbase Ward   
4         199  2078  43168.0  Nairobi  Kamukunji  Sub County  Airbase Ward   

           uid        scuid         cuid  \
0  t4Suo8Enc7T  qoLIT7y5f5c  jkG3zaihdSs   
1  t4Suo8Enc7T  qoLIT7y5f5c  jkG3zaihdSs   
2  t4Suo8Enc7T  qoLIT7y5f5c  jkG3zaihdSs   
3  t4Suo8Enc7T  qoLIT7y5f5c  jkG3zaihdSs   
4  t4Suo8Enc7T  qoLIT7y5f5c  jkG3zaihdSs   

                                            geometry  ...   population_y  \
0  POLYGON ((263765.5843998674 9860350.470779378,...  ...  105433.255157   
1  POLYGON ((263765.5843998674 9860350.470779378,...  ...  105433.255157   
2  POLYGON ((263765.5843998674 9860350